# Modelos finales
## Identificación de menciones de entidades biomédicas en resúmenes de investigación

In [1]:
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import pandas as pd
import nltk
import re

### Preprocesamiento

In [2]:
# download necessary nltk resources
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/melissa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/melissa/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# function to clean and preprocess text
def preprocessText(text):
    # convert to lowercase
    text = text.lower()
    
    # remove non-alphabetic characters
    text = re.sub(r'[^a-z\s]', '', text)
    
    # tokenize by spaces
    tokens = text.split()
    
    # remove stopwords
    stopWords = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stopWords]
    
    # lemmatization
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    return ' '.join(tokens)

In [4]:
# path of the data
dataPath = '../data/'

# load data
abstractsTrain = pd.read_csv(dataPath + 'abstracts_train.csv', on_bad_lines='skip', delimiter='\t')
entitiesTrain = pd.read_csv(dataPath + 'entities_train.csv', on_bad_lines='skip', delimiter='\t')
relationsTrain = pd.read_csv(dataPath + 'relations_train.csv', on_bad_lines='skip', delimiter='\t')
abstractsTest = pd.read_csv(dataPath + 'abstracts_test.csv', on_bad_lines='skip', delimiter='\t')

In [5]:
# apply the preprocessing function to the abstract text column
abstractsTrain['cleanAbstract'] = abstractsTrain['abstract'].apply(preprocessText)
abstractsTest['cleanAbstract'] = abstractsTest['abstract'].apply(preprocessText)

# show the cleaned texts
print(abstractsTrain[['abstract', 'cleanAbstract']].head())
print(abstractsTest[['abstract', 'cleanAbstract']].head())

                                            abstract  \
0  We report on a new allele at the arylsulfatase...   
1  Classical phenylketonuria is an autosomal rece...   
2  The metabolism of the cardioselective beta-blo...   
3  Previous experiments in this laboratory have s...   
4  Eighty unrelated individuals with Duchenne mus...   

                                       cleanAbstract  
0  report new allele arylsulfatase arsa locus cau...  
1  classical phenylketonuria autosomal recessive ...  
2  metabolism cardioselective betablocker metopro...  
3  previous experiment laboratory shown microinje...  
4  eighty unrelated individual duchenne muscular ...  
                                            abstract  \
0  The effect of induced hypertension instituted ...   
1  A linkage study in 30 Becker muscular dystroph...   
2  The effects of a 6-hour infusion with haloperi...   
3  Fragments of the adrenoleukodystrophy (ALD) cD...   
4  The ability to scan a large gene rapidly and a... 

In [6]:
# group entities by abstract_id
entities_grouped = entitiesTrain.groupby('abstract_id')['type'].apply(list).reset_index()

# merge the cleaned abstracts with their corresponding entities by 'abstract_id'
data_merged = pd.merge(abstractsTrain, entities_grouped, how='inner', on='abstract_id')

### Implementación de modelos

In [7]:
from sklearn.model_selection import train_test_split

# dataset splitting, 80% for training and 20% for testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    data_merged['cleanAbstract'], data_merged['type'], test_size=0.2, random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42  # 0.25 * 0.8 = 0.2 del total
)

# sizes of the splits
print(f'Training set for validation size: {len(X_train_val)}')
print(f'Test set for prediccitions size: {len(X_test)}')
print(f'Training set size: {len(X_train)}')
print(f'Validation set size: {len(X_val)}')

Training set for validation size: 320
Test set for prediccitions size: 80
Training set size: 240
Validation set size: 80


#### Support Vector Machines (SVM)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train)
y_val_encoded = mlb.transform(y_val)

vectorizer = TfidfVectorizer()
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)

svm_model = OneVsRestClassifier(SVC(kernel='linear', probability=True))

svm_model.fit(X_train_tfidf, y_train_encoded)

y_pred = svm_model.predict(X_val_tfidf)

print(classification_report(y_val_encoded, y_pred, target_names=mlb.classes_))


                            precision    recall  f1-score   support

                  CellLine       1.00      0.10      0.18        10
            ChemicalEntity       0.75      0.86      0.80        50
DiseaseOrPhenotypicFeature       0.99      1.00      0.99        79
         GeneOrGeneProduct       0.88      0.97      0.92        68
             OrganismTaxon       0.85      1.00      0.92        68
           SequenceVariant       0.92      0.94      0.93        35

                 micro avg       0.88      0.94      0.91       310
                 macro avg       0.90      0.81      0.79       310
              weighted avg       0.89      0.94      0.90       310
               samples avg       0.88      0.94      0.90       310



#### Graph Convolutional Networks (GCN)

In [9]:
import networkx as nx
import numpy as np

G = nx.Graph()

for idx, (abstract, entities) in enumerate(zip(X_train, y_train)):
    for entity in entities:
        unique_entity_id = f"{idx}_{entity}"
        
        G.add_node(unique_entity_id, abstract_id=idx, entity=abstract, x=np.random.rand(128))

for idx, entities in enumerate(y_train):
    for i in range(len(entities)):
        for j in range(i + 1, len(entities)):
            node_id_1 = f"{idx}_{entities[i]}"
            node_id_2 = f"{idx}_{entities[j]}"
            G.add_edge(node_id_1, node_id_2, relation_type="related")

standard_attributes = {'abstract_id': None, 'entity': None}
for node in G.nodes:
    for attr, default_value in standard_attributes.items():
        if attr not in G.nodes[node]:
            G.nodes[node][attr] = default_value


In [10]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
from torch_geometric.utils import from_networkx
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

data = from_networkx(G)

data.x = torch.tensor([G.nodes[node]['x'] for node in G.nodes], dtype=torch.float32)

mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(y_train)  
y_full = np.zeros((data.num_nodes, len(mlb.classes_)))

for idx, entities in enumerate(y_train):
    y_encoded = mlb.transform([entities])[0]
    y_full[idx] = y_encoded

data.y = torch.tensor(y_full, dtype=torch.float32)

class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x 

modelGCN = GCN(in_channels=128, hidden_channels=16, out_channels=len(mlb.classes_))
optimizer = torch.optim.Adam(modelGCN.parameters(), lr=0.01, weight_decay=5e-4)

def train():
    modelGCN.train()
    optimizer.zero_grad()
    out = modelGCN(data.x, data.edge_index)
    loss = F.binary_cross_entropy_with_logits(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()

for epoch in range(500):
    loss = train()
    print(f'Epoch {epoch}: Loss {loss:.4f}')


/Users/melissa/Desktop/UVG/dataScience/Proyecto02-CC3084/.venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/melissa/Desktop/UVG/dataScience/Proyecto02-CC3084/.venv/lib/python3.8/site-packages/torch_geometric/utils/convert.py:278: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:281.)
  data_dict[key] = torch.as_tensor(value)


Epoch 0: Loss 0.6813
Epoch 1: Loss 0.6007
Epoch 2: Loss 0.5361
Epoch 3: Loss 0.5090
Epoch 4: Loss 0.4974
Epoch 5: Loss 0.4838
Epoch 6: Loss 0.4684
Epoch 7: Loss 0.4561
Epoch 8: Loss 0.4495
Epoch 9: Loss 0.4478
Epoch 10: Loss 0.4464
Epoch 11: Loss 0.4426
Epoch 12: Loss 0.4373
Epoch 13: Loss 0.4331
Epoch 14: Loss 0.4309
Epoch 15: Loss 0.4299
Epoch 16: Loss 0.4292
Epoch 17: Loss 0.4286
Epoch 18: Loss 0.4279
Epoch 19: Loss 0.4274
Epoch 20: Loss 0.4268
Epoch 21: Loss 0.4257
Epoch 22: Loss 0.4242
Epoch 23: Loss 0.4231
Epoch 24: Loss 0.4223
Epoch 25: Loss 0.4218
Epoch 26: Loss 0.4213
Epoch 27: Loss 0.4204
Epoch 28: Loss 0.4199
Epoch 29: Loss 0.4194
Epoch 30: Loss 0.4189
Epoch 31: Loss 0.4184
Epoch 32: Loss 0.4177
Epoch 33: Loss 0.4170
Epoch 34: Loss 0.4164
Epoch 35: Loss 0.4158
Epoch 36: Loss 0.4153
Epoch 37: Loss 0.4148
Epoch 38: Loss 0.4144
Epoch 39: Loss 0.4140
Epoch 40: Loss 0.4135
Epoch 41: Loss 0.4130
Epoch 42: Loss 0.4125
Epoch 43: Loss 0.4120
Epoch 44: Loss 0.4114
Epoch 45: Loss 0.410

#### Bidirectional Encoder Representations from Transformers (BERT)

In [11]:
from transformers import BertTokenizer, BertForSequenceClassification
num_classes = 6
tokenizerBert = BertTokenizer.from_pretrained("bert-base-uncased")
modelBERT = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=num_classes)

mlb = MultiLabelBinarizer()
y_train_encoded = mlb.fit_transform(y_train)
y_val_encoded = mlb.transform(y_val)

def preprocess_texts_BERT(texts, labels):
    inputs = tokenizerBert(
        texts, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    inputs["labels"] = torch.tensor(labels, dtype=torch.float)
    return inputs

train_texts = list(X_train)
train_labels = y_train_encoded
train_data = preprocess_texts_BERT(train_texts, train_labels)

val_texts = list(X_val)
val_labels = y_val_encoded
val_data = preprocess_texts_BERT(val_texts, val_labels)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [12]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: tensor[idx] for key, tensor in self.encodings.items()}

train_dataset = TextDataset(train_data)
val_dataset = TextDataset(val_data)

In [13]:
import os

os.environ["TRANSFORMERS_NO_TF"] = "1"
from transformers import Trainer, TrainingArguments


training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
)

trainer = Trainer(
    model=modelBERT,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

trainer.train()

/Users/melissa/Desktop/UVG/dataScience/Proyecto02-CC3084/.venv/lib/python3.8/site-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
 11%|█         | 10/90 [00:08<00:32,  2.46it/s]

{'loss': 0.7152, 'grad_norm': 3.2656641006469727, 'learning_rate': 1.0000000000000002e-06, 'epoch': 0.33}


 22%|██▏       | 20/90 [00:11<00:24,  2.83it/s]

{'loss': 0.6749, 'grad_norm': 2.210542678833008, 'learning_rate': 2.0000000000000003e-06, 'epoch': 0.67}


 33%|███▎      | 30/90 [00:15<00:20,  2.86it/s]

...

                                               
100%|██████████| 90/90 [00:44<00:00,  2.84it/s]

{'eval_loss': 0.4525209069252014, 'eval_runtime': 0.8519, 'eval_samples_per_second': 93.902, 'eval_steps_per_second': 11.738, 'epoch': 3.0}


100%|██████████| 90/90 [00:46<00:00,  1.95it/s]

{'train_runtime': 46.5871, 'train_samples_per_second': 15.455, 'train_steps_per_second': 1.932, 'train_loss': 0.5833390871683757, 'epoch': 3.0}


TrainOutput(global_step=90, training_loss=0.5833390871683757, metrics={'train_runtime': 46.5871, 'train_samples_per_second': 15.455, 'train_steps_per_second': 1.932, 'total_flos': 47361690869760.0, 'train_loss': 0.5833390871683757, 'epoch': 3.0})

In [14]:
def preprocess_text_for_prediction(text):
    inputs = tokenizerBert(
        text, padding=True, truncation=True, max_length=128, return_tensors="pt"
    )
    return inputs["input_ids"], inputs["attention_mask"]

### Predicciones

In [15]:
import re

def extract_entity_ids(abstract):
    matched_entity_ids = []
    tokens = re.findall(r'\w+', abstract.lower())

    for node_id, node_data in G.nodes(data=True):
        entity_text = node_data.get('entity', '')
        if entity_text:
            entity_tokens = set(re.findall(r'\w+', entity_text.lower()))
            if entity_tokens.intersection(tokens):  
                matched_entity_ids.append(node_id)
    return matched_entity_ids

In [16]:
def predict_entities_svm(abstract):
    tfidf_sequence = vectorizer.transform([abstract])
    prediction = svm_model.predict(tfidf_sequence)
    predicted_entities = mlb.inverse_transform(prediction)
    return predicted_entities[0] 

def predict_entities_gcn(abstract):
    entity_ids = extract_entity_ids(abstract)
    subgraph_nx = G.subgraph(entity_ids)
    subgraph_data = from_networkx(subgraph_nx)
    if subgraph_data.x is not None and subgraph_data.x.numel() > 0:
        subgraph_data.x = subgraph_data.x.float()
        
        modelGCN.eval()
        with torch.no_grad():
            out = modelGCN(subgraph_data.x, subgraph_data.edge_index)
        
        predicted_classes = (out > 0.5).cpu().numpy()
        predicted_entities = mlb.inverse_transform(predicted_classes)
        return predicted_entities[0]
    return ()

def predict_entities_bert(abstract):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    
    input_ids, attention_mask = preprocess_text_for_prediction(abstract)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)
    
    modelBERT.to(device)
    modelBERT.eval()
    with torch.no_grad():
        outputs = modelBERT(input_ids, attention_mask=attention_mask)
    
    predicted_classes = (outputs.logits > 0.5).cpu().numpy()
    predicted_entities = mlb.inverse_transform(predicted_classes)
    
    return predicted_entities[0] 

In [17]:
abstract_example = "metachromatic"

print("SVM Prediction:", predict_entities_svm(abstract_example))
print("GCN Prediction:", predict_entities_gcn(abstract_example))
print("BERT Prediction:", predict_entities_bert(abstract_example))

SVM Prediction: ('ChemicalEntity', 'DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct', 'OrganismTaxon')
GCN Prediction: ()
BERT Prediction: ('DiseaseOrPhenotypicFeature', 'GeneOrGeneProduct')


### Evaluación de los modelos

In [18]:
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

In [19]:
def get_model_predictions(abstracts, model_predict_function):
    predictions = []
    for abstract in abstracts:
        predicted_entities = model_predict_function(abstract)
        predictions.append(predicted_entities)
    return predictions

# Obtener predicciones reales de cada modelo
y_pred_svm = get_model_predictions(X_test, predict_entities_svm)
y_pred_gcn = get_model_predictions(X_test, predict_entities_gcn)
y_pred_bert = get_model_predictions(X_test, predict_entities_bert)

In [20]:
def calculate_model_metrics(y_true, y_pred, model_name):
    # Convertir `y_true` y `y_pred` a formato binario
    mlb = MultiLabelBinarizer()
    y_true_bin = mlb.fit_transform(y_true)
    y_pred_bin = mlb.transform(y_pred)

    # Calcular las métricas con el formato binario
    f1 = f1_score(y_true_bin, y_pred_bin, average='micro')
    seq_label_accuracy = accuracy_score(y_true_bin, y_pred_bin)
    roc_auc = roc_auc_score(y_true_bin, y_pred_bin, average="micro")
    
    return {"Model": model_name, "F1 Score (Micro)": f1, "Sequence Label Accuracy": seq_label_accuracy, "ROC-AUC": roc_auc}

In [21]:
# Calcular métricas para cada modelo
metrics = [
    calculate_model_metrics(y_test, y_pred_svm, "SVM"),
    calculate_model_metrics(y_test, y_pred_gcn, "GCN"),
    calculate_model_metrics(y_test, y_pred_bert, "BERT")
]

# Mostrar métricas
metrics_df = pd.DataFrame(metrics)
print(metrics_df)

  Model  F1 Score (Micro)  Sequence Label Accuracy   ROC-AUC
0   SVM          0.901716                   0.3875  0.841907
1   GCN          0.768670                   0.0500  0.756629
2  BERT          0.833898                   0.1875  0.795719


### Exportación de modelos

In [ ]:
# save models to pkls
import joblib

joblib.dump(svm_model, 'models/svm_model.pkl')                 #svm
joblib.dump(modelGCN.state_dict(), "models/gcn_model.pkl")     #gcn

['models/bert_model.pkl']